<a href="https://colab.research.google.com/github/obsdnx/local-llm-/blob/main/cloud/train_8b_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train Forge 8B on Google Colab Pro

**Before you start**
1. Menu **Runtime → Change runtime type → A100 GPU** (or **L4 GPU**). Click **Save**.
2. Menu **Runtime → Run all**.
3. When asked, allow access to Google Drive (the finished model is saved there).

Takes about 1–2 hours. Keep this browser tab open.

**When it's done**
1. Download the `.gguf` file from Google Drive folder **forge**.
2. On your Mac, in the project folder: `bash cloud/install_on_mac.sh ~/Downloads/<file>.gguf`
3. Chat: `uv run chat.py --model forge-8b`
4. Here in Colab: **Runtime → Disconnect and delete runtime** (stops using compute units).

## 1. Check the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 2. Install Unsloth (commands from Unsloth's official Qwen3 Colab notebook)

In [2]:
%%capture
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

## 3. Connect Google Drive and download the project

In [4]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/forge
!rm -rf /content/forge && git clone --depth 1 https://github.com/obsdnx/local-llm-.git /content/forge
%cd /content/forge
!wc -l training/*.jsonl | tail -1

Mounted at /content/drive
Cloning into '/content/forge'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 38 (delta 0), reused 33 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 316.72 KiB | 1.70 MiB/s, done.
/content/forge
   489 total


## 4. Train (the long step)
Shows the loss every 5 steps and tests on held-out examples every 20 steps. Lower `eval_loss` is better; the best checkpoint is kept automatically.

In [ ]:
!python cloud/train_8b.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4 (16 GB) - small GPU settings
==((====))==  Unsloth 2026.10.8: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading checkpoint shards: 100% 2/2 [00:29<00:00, 14.98s/it]
Unsloth 2026.10.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).
c_code.jsonl: 14 examples
c_memory.jsonl: 45 examples
c_systems.jsonl: 45 examples
cybersecurity.jsonl: 14 examples
linux.jsonl: 14 examples
l

## 5. Save the results to Google Drive

In [ ]:
!find /content/forge -name "*.gguf" -exec cp -v {} /content/drive/MyDrive/forge/ \;
!cp -r /content/forge/forge-8b-lora /content/drive/MyDrive/forge/
!ls -lh /content/drive/MyDrive/forge